<a href="https://colab.research.google.com/github/Sarkis55/Datathon2026/blob/main/Datathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# We Start Here!
#### Slide Link: [Link](https://www.canva.com/design/DAHD775lg0M/bJVGthE0XBZCr_9CL_MAAA/edit?utm_content=DAHD775lg0M&utm_campaign=designshare&utm_medium=link2&utm_source=sharebutton)

In [ ]:
# =========================================================
# Graduate Assignment: Gender Pay Gap and Prior Pay Analysis
# English version for Google Colab
# Includes figure export
# =========================================================

# =========================
# 0. Install and import
# =========================
!pip -q install pandas numpy matplotlib seaborn statsmodels scipy

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

sns.set_context("talk")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Create output folder for figures and tables
output_dir = "/content/output_figures"
os.makedirs(output_dir, exist_ok=True)

# =========================
# 1. Upload file from user
# =========================
from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

file_name = list(uploaded.keys())[0]
df_raw = pd.read_csv(file_name)

print("Uploaded file:", file_name)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

# =========================
# 2. Inspect data
# =========================
print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

print("\nMissing value percentage:")
display((df_raw.isna().mean().sort_values(ascending=False) * 100).map(lambda x: f"{x:.2f}%"))

# -----------------------------
# Check and handle duplicates
# -----------------------------
print("\nChecking for duplicate rows...")
duplicate_count = df_raw.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

if duplicate_count > 0:
    df_raw = df_raw.drop_duplicates()
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

print("Shape after duplicate handling:", df_raw.shape)

# =========================
# 3. Data cleaning
# =========================
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

numeric_cols = [
    "PUBID_1997", "SAMPLE_RACE_1997", "SAMPLE_SEX_1997", "Year",
    "Employed", "TENURE", "HRLY_WAGE", "HRLY_COMP", "HRS_WRK",
    "UID", "Code_1990", "marital_status", "HGC", "Region"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

date_cols = ["DOB", "Interview_Date", "StartDate", "StopDate"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# =========================
# 3A. Drop useless columns
# =========================
drop_cols = []

if "index_col" in df.columns:
    drop_cols.append("index_col")

# Drop Employed after filtering since it becomes constant
# Keep it for now because it is used in the employment filter below

if len(drop_cols) > 0:
    df = df.drop(columns=drop_cols)
    print("Dropped columns:", drop_cols)

# Keep employed observations with valid hourly wage
df = df[df["Employed"] == 1].copy()
df = df[df["HRLY_WAGE"].notna()].copy()
df = df[df["HRLY_WAGE"] > 0].copy()

# Now Employed is constant, so drop it
if "Employed" in df.columns:
    df = df.drop(columns=["Employed"])

# Trim extreme wages using 1st and 99th percentiles
wage_low = df["HRLY_WAGE"].quantile(0.01)
wage_high = df["HRLY_WAGE"].quantile(0.99)
df = df[(df["HRLY_WAGE"] >= wage_low) & (df["HRLY_WAGE"] <= wage_high)].copy()

# Keep valid weekly hours if available
if "HRS_WRK" in df.columns:
    df = df[(df["HRS_WRK"].isna()) | ((df["HRS_WRK"] > 0) & (df["HRS_WRK"] <= 120))].copy()

# Keep nonnegative tenure if available
if "TENURE" in df.columns:
    df = df[(df["TENURE"].isna()) | (df["TENURE"] >= 0)].copy()

print("Cleaned sample shape:", df.shape)

# =========================
# 4. Feature engineering
# =========================

# Assumption: 1 = Male, 2 = Female
df["female"] = np.where(df["SAMPLE_SEX_1997"] == 2, 1, 0)

# =========================
# 4A. Create date based features
# =========================
if "DOB" in df.columns and "Interview_Date" in df.columns:
    df["age"] = (df["Interview_Date"] - df["DOB"]).dt.days / 365.25
else:
    df["age"] = np.nan

df["age_sq"] = df["age"] ** 2

if "Interview_Date" in df.columns:
    df["interview_year"] = df["Interview_Date"].dt.year
    df["interview_month"] = df["Interview_Date"].dt.month

if "StartDate" in df.columns:
    df["start_year"] = df["StartDate"].dt.year
    df["start_month"] = df["StartDate"].dt.month

if "StopDate" in df.columns:
    df["stop_year"] = df["StopDate"].dt.year
    df["stop_month"] = df["StopDate"].dt.month

if "StartDate" in df.columns and "StopDate" in df.columns:
    df["job_duration_days"] = (df["StopDate"] - df["StartDate"]).dt.days
else:
    df["job_duration_days"] = np.nan

if "StartDate" in df.columns and "Interview_Date" in df.columns:
    df["days_since_start_to_interview"] = (df["Interview_Date"] - df["StartDate"]).dt.days
else:
    df["days_since_start_to_interview"] = np.nan

# =========================
# 4B. Transform wage variables
# =========================
df["ln_wage"] = np.log(df["HRLY_WAGE"])

if "HRLY_COMP" in df.columns:
    df["ln_comp"] = np.where(df["HRLY_COMP"] > 0, np.log(df["HRLY_COMP"]), np.nan)

# Optional wage ratio
if "HRLY_COMP" in df.columns:
    df["wage_to_comp_ratio"] = np.where(df["HRLY_COMP"] > 0, df["HRLY_WAGE"] / df["HRLY_COMP"], np.nan)

# =========================
# 4C. Treat coded numeric categories as categorical
# =========================
coded_cat_cols = ["SAMPLE_RACE_1997", "SAMPLE_SEX_1997", "marital_status", "Region"]

for col in coded_cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("Int64").astype("category")

race_map = {
    1: "Hispanic",
    2: "Black",
    3: "Non-Black/Non-Hispanic",
    4: "Mixed/Other"
}
df["race_label"] = df["SAMPLE_RACE_1997"].astype("object").map(race_map).fillna("Unknown")

sex_map = {
    1: "Male",
    2: "Female"
}
df["sex_label"] = df["SAMPLE_SEX_1997"].astype("object").map(sex_map).fillna("Unknown")

region_map = {
    1: "Northeast",
    2: "North Central",
    3: "South",
    4: "West"
}
df["region_label"] = df["Region"].astype("object").map(region_map).fillna("Unknown")

# Optional marital status label
marital_map = {
    0: "Not married",
    1: "Married"
}
df["marital_label"] = df["marital_status"].astype("object").map(marital_map).fillna("Other/Unknown")

# =========================
# 4D. Encode large text categories
# =========================
text_cat_cols = ["Occupation_Group2", "Industry_Group", "Occupation", "Industry"]

for col in text_cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str).str.strip()
        df[col] = df[col].replace("", "Unknown")
        df[col] = df[col].astype("category")

# Optional numeric codes for export or ML use
for col in text_cat_cols:
    if col in df.columns:
        df[f"{col}_code"] = df[col].cat.codes

# Sort before building prior wage
sort_cols = ["PUBID_1997", "Year"]
if "Interview_Date" in df.columns:
    sort_cols.append("Interview_Date")

df = df.sort_values(sort_cols).copy()

# Prior wage
df["prior_wage"] = df.groupby("PUBID_1997")["HRLY_WAGE"].shift(1)
df["ln_prior_wage"] = np.where(df["prior_wage"] > 0, np.log(df["prior_wage"]), np.nan)

df["prior_year"] = df.groupby("PUBID_1997")["Year"].shift(1)
df["year_gap"] = df["Year"] - df["prior_year"]

df_lag = df[df["prior_wage"].notna()].copy()
df_lag = df_lag[df_lag["prior_wage"] > 0].copy()
df_lag = df_lag[(df_lag["year_gap"].isna()) | (df_lag["year_gap"] <= 3)].copy()

print("Lag sample shape:", df_lag.shape)

# =========================
# 5. Summary statistics
# =========================
print("\nOverall sample summary:")
display(df[["HRLY_WAGE", "ln_wage", "HRS_WRK", "TENURE", "HGC", "age", "Year"]].describe().T)

print("\nGender distribution:")
display(df["sex_label"].value_counts(dropna=False).to_frame("count"))

print("\nRace distribution:")
display(df["race_label"].value_counts(dropna=False).to_frame("count"))

print("\nRegion distribution:")
display(df["region_label"].value_counts(dropna=False).to_frame("count"))

print("\nNumber of distinct individuals:")
print(df["PUBID_1997"].nunique())

year_gender = (
    df.groupby(["Year", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"),
           median_wage=("HRLY_WAGE", "median"),
           mean_ln_wage=("ln_wage", "mean"),
           n=("HRLY_WAGE", "size"))
)

print("\nYear by gender wage summary:")
display(year_gender.head(20))

# =========================
# 6. Mean difference test
# =========================
male_wage = df.loc[df["female"] == 0, "HRLY_WAGE"].dropna()
female_wage = df.loc[df["female"] == 1, "HRLY_WAGE"].dropna()

t_stat, p_value = stats.ttest_ind(male_wage, female_wage, equal_var=False, nan_policy="omit")

print("\nMean wage difference test:")
print("Male mean wage  :", round(male_wage.mean(), 3))
print("Female mean wage:", round(female_wage.mean(), 3))
print("T-statistic     :", round(t_stat, 4))
print("P-value         :", p_value)

raw_gap_pct = 100 * (female_wage.mean() / male_wage.mean() - 1)
print("Raw female vs male wage gap (%):", round(raw_gap_pct, 2))

# =========================
# 7. Export figures
# =========================
# Figure 1: Wage distribution by gender
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="HRLY_WAGE", hue="sex_label", bins=50, kde=True, stat="density", common_norm=False)
plt.title("Hourly Wage Distribution by Gender")
plt.xlabel("Hourly Wage")
plt.ylabel("Density")
plt.tight_layout()
plt.savefig(f"{output_dir}/figure_1_wage_distribution_by_gender.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Figure 2: Average wage over time by gender
plt.figure(figsize=(12, 6))
sns.lineplot(data=year_gender, x="Year", y="mean_wage", hue="sex_label", marker="o")
plt.title("Average Hourly Wage by Gender Over Time")
plt.xlabel("Year")
plt.ylabel("Average Hourly Wage")
plt.tight_layout()
plt.savefig(f"{output_dir}/figure_2_average_wage_over_time_by_gender.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Figure 3: Boxplot of hourly wage by gender
plot_df = df[df["HRLY_WAGE"] <= df["HRLY_WAGE"].quantile(0.95)].copy()
plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_df, x="sex_label", y="HRLY_WAGE")
plt.title("Hourly Wage by Gender")
plt.xlabel("Gender")
plt.ylabel("Hourly Wage")
plt.tight_layout()
plt.savefig(f"{output_dir}/figure_3_boxplot_hourly_wage_by_gender.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Figure 4: Prior wage vs current wage
plot_lag = df_lag[
    (df_lag["HRLY_WAGE"] <= df_lag["HRLY_WAGE"].quantile(0.99)) &
    (df_lag["prior_wage"] <= df_lag["prior_wage"].quantile(0.99))
].copy()

sample_size = min(8000, len(plot_lag))
plot_sample = plot_lag.sample(sample_size, random_state=42) if sample_size > 0 else plot_lag.copy()

plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_sample, x="prior_wage", y="HRLY_WAGE", hue="sex_label", alpha=0.35)
plt.title("Prior Wage and Current Wage")
plt.xlabel("Prior Wage")
plt.ylabel("Current Wage")
plt.tight_layout()
plt.savefig(f"{output_dir}/figure_4_prior_wage_vs_current_wage.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# 8. Regression models
# =========================
# Model 1
m1 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\nModel 1: Gender pay gap with controls")
print(m1.summary())

# Model 2
m2 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\nModel 2: Add marital status")
print(m2.summary())

# Model 3
m3 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\nModel 3: Add occupation and industry")
print(m3.summary())

# Model 4
m4 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage + female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\nModel 4: Current wage on prior wage")
print(m4.summary())

# Model 5
m5 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\nModel 5: Interaction between prior wage and female")
print(m5.summary())

# Model 6
df_lag["post_2018"] = np.where(df_lag["Year"] >= 2018, 1, 0)

m6 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * post_2018 + female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\nModel 6: Prior wage relationship after 2018")
print(m6.summary())

# Model 7
m7 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage + female * post_2018 + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\nModel 7: Female effect before and after 2018")
print(m7.summary())

# Model 8
m8 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * female * post_2018 + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\nModel 8: Triple interaction model")
print(m8.summary())

# =========================
# 9. Heterogeneity analysis
# =========================
occupation_gap = (
    df.groupby(["Occupation_Group2", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"), n=("HRLY_WAGE", "size"))
)

occupation_pivot = occupation_gap.pivot(index="Occupation_Group2", columns="sex_label", values="mean_wage")
occupation_pivot["female_minus_male"] = occupation_pivot.get("Female", np.nan) - occupation_pivot.get("Male", np.nan)
occupation_pivot["female_to_male_ratio"] = occupation_pivot.get("Female", np.nan) / occupation_pivot.get("Male", np.nan)
occupation_pivot = occupation_pivot.sort_values("female_to_male_ratio")

print("\nOccupation-level wage comparison:")
display(occupation_pivot.head(20))

industry_gap = (
    df.groupby(["Industry_Group", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"), n=("HRLY_WAGE", "size"))
)

industry_pivot = industry_gap.pivot(index="Industry_Group", columns="sex_label", values="mean_wage")
industry_pivot["female_minus_male"] = industry_pivot.get("Female", np.nan) - industry_pivot.get("Male", np.nan)
industry_pivot["female_to_male_ratio"] = industry_pivot.get("Female", np.nan) / industry_pivot.get("Male", np.nan)
industry_pivot = industry_pivot.sort_values("female_to_male_ratio")

print("\nIndustry-level wage comparison:")
display(industry_pivot.head(20))

plot_occ = occupation_pivot.dropna(subset=["female_to_male_ratio"]).reset_index()
plot_occ = plot_occ.sort_values("female_to_male_ratio").tail(15)

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_occ, y="Occupation_Group2", x="female_to_male_ratio")
plt.axvline(1.0, linestyle="--")
plt.title("Female-to-Male Average Wage Ratio by Occupation Group")
plt.xlabel("Female / Male Wage Ratio")
plt.ylabel("Occupation Group")
plt.tight_layout()
plt.savefig(f"{output_dir}/figure_5_female_to_male_ratio_by_occupation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# =========================
# 10. Save regression summary
# =========================
def coef_to_pct(beta):
    if pd.isna(beta):
        return np.nan
    return (np.exp(beta) - 1) * 100

results_table = pd.DataFrame({
    "model": ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"],
    "female_coef": [
        m1.params.get("female", np.nan),
        m2.params.get("female", np.nan),
        m3.params.get("female", np.nan),
        m4.params.get("female", np.nan),
        m5.params.get("female", np.nan),
        m6.params.get("female", np.nan),
        m7.params.get("female", np.nan),
        m8.params.get("female", np.nan),
    ],
    "ln_prior_wage_coef": [
        np.nan,
        np.nan,
        np.nan,
        m4.params.get("ln_prior_wage", np.nan),
        m5.params.get("ln_prior_wage", np.nan),
        m6.params.get("ln_prior_wage", np.nan),
        m7.params.get("ln_prior_wage", np.nan),
        m8.params.get("ln_prior_wage", np.nan),
    ],
    "nobs": [m.nobs for m in [m1, m2, m3, m4, m5, m6, m7, m8]],
    "r_squared": [m.rsquared for m in [m1, m2, m3, m4, m5, m6, m7, m8]]
})

results_table["female_pct_effect"] = results_table["female_coef"].apply(coef_to_pct)

display(results_table)

results_table.to_csv("/content/regression_summary.csv", index=False)
occupation_pivot.to_csv("/content/occupation_gap_summary.csv")
industry_pivot.to_csv("/content/industry_gap_summary.csv")

# =========================
# 11. Save cleaned data
# =========================
df.to_csv("/content/cleaned_main_sample.csv", index=False)
df_lag.to_csv("/content/cleaned_lag_sample.csv", index=False)

# =========================
# 12. Final output message
# =========================
print("\nSaved output files:")
print("/content/cleaned_main_sample.csv")
print("/content/cleaned_lag_sample.csv")
print("/content/regression_summary.csv")
print("/content/occupation_gap_summary.csv")
print("/content/industry_gap_summary.csv")
print(f"{output_dir}/figure_1_wage_distribution_by_gender.png")
print(f"{output_dir}/figure_2_average_wage_over_time_by_gender.png")
print(f"{output_dir}/figure_3_boxplot_hourly_wage_by_gender.png")
print(f"{output_dir}/figure_4_prior_wage_vs_current_wage.png")
print(f"{output_dir}/figure_5_female_to_male_ratio_by_occupation.png")

print("\nInterpretation guide:")
print("\nInterpretation guide:")
print("""
1. If the coefficient on female is significantly negative, women earn less than men after controls.
2. If ln_prior_wage is significantly positive, prior wage strongly predicts current wage.
3. If ln_prior_wage:female is significant, the prior wage effect differs by gender.
4. If ln_prior_wage:post_2018 is negative, the prior-current wage relationship may have weakened after 2018.
5. If the triple interaction is significant, the role of prior wage for women changed after 2018.
""")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.4 MB/s eta 0:00:00


# 2nd Part_Data Mining by Sammy

In [ ]:
# =========================================================
# Colab: Load graduate-full.csv.zip directly from GitHub
# Repo: Sarkis55/Datathon2026
# =========================================================

!pip -q install pandas numpy requests

import os
import io
import zipfile
import requests
import pandas as pd
import numpy as np

# -----------------------------
# 1. GitHub raw ZIP URL
# -----------------------------
ZIP_URL = "https://raw.githubusercontent.com/Sarkis55/Datathon2026/main/graduate-full.csv.zip"

# Local paths in Colab
DOWNLOAD_DIR = "/content/datathon_data"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "graduate-full.csv.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

# -----------------------------
# 2. Download ZIP from GitHub
# -----------------------------
print("Downloading ZIP from GitHub...")
response = requests.get(ZIP_URL, timeout=120)
response.raise_for_status()

with open(ZIP_PATH, "wb") as f:
    f.write(response.content)

print(f"ZIP saved to: {ZIP_PATH}")
print(f"Downloaded size: {os.path.getsize(ZIP_PATH):,} bytes")

# -----------------------------
# 3. Extract ZIP
# -----------------------------
print("Extracting ZIP...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)
    extracted_files = zf.namelist()

print("Extracted files:")
for f in extracted_files:
    print("-", f)

# -----------------------------
# 4. Find CSV automatically
# -----------------------------
csv_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file was found after extracting the ZIP.")

if len(csv_files) > 1:
    print("Multiple CSV files found. The first one will be used:")
    for path in csv_files:
        print("-", path)

CSV_PATH = csv_files[0]
print(f"Using CSV file: {CSV_PATH}")

# -----------------------------
# 5. Read CSV
# -----------------------------
print("Reading CSV into pandas...")
df_raw = pd.read_csv(CSV_PATH)

print("Raw shape:", df_raw.shape)
display(df_raw.head())

# -----------------------------
# 6. Optional: basic inspection
# -----------------------------
print("\nColumn names:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

print("\nMissing value percentage:")
display((df_raw.isna().mean().sort_values(ascending=False) * 100).round(2))

# -----------------------------
# 7. Optional: save a working copy
# -----------------------------
WORKING_CSV_PATH = "/content/graduate-full.csv"
df_raw.to_csv(WORKING_CSV_PATH, index=False)
print(f"\nWorking CSV saved to: {WORKING_CSV_PATH}")
# =========================
# 1. Load ZIP from GitHub
# =========================
import os
import zipfile
import requests
import pandas as pd

ZIP_URL = "https://raw.githubusercontent.com/Sarkis55/Datathon2026/main/graduate-full.csv.zip"

DOWNLOAD_DIR = "/content/datathon_data"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "graduate-full.csv.zip")
EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "extracted")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

response = requests.get(ZIP_URL, timeout=120)
response.raise_for_status()

with open(ZIP_PATH, "wb") as f:
    f.write(response.content)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

csv_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

if not csv_files:
    raise FileNotFoundError("No CSV file found inside the ZIP.")

file_path = csv_files[0]
df_raw = pd.read_csv(file_path)

print("Using file:", file_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

ZIP saved to: /content/datathon_data/graduate-full.csv.zip
Downloaded size: 6,433,452 bytes
Extracting ZIP...
Extracted files:
- graduate-full.csv
- __MACOSX/._graduate-full.csv
Multiple CSV files found. The first one will be used:
- /content/datathon_data/extracted/graduate-full.csv
- /content/datathon_data/extracted/__MACOSX/._graduate-full.csv
Using CSV file: /content/datathon_data/extracted/graduate-full.csv
Reading CSV into pandas...
Raw shape: (194545, 26)


,PUBID_1997,DOB,SAMPLE_RACE_1997,SAMPLE_SEX_1997,Interview_Date,Year,Employed,StartDate,StopDate,IND,OCC,TENURE,HRLY_WAGE,HRLY_COMP,HRS_WRK,UID,Code_1990,Occupation,Occupation_Group,Occupation_Group2,Industry,Industry_Group,marital_status,HGC,Region,index_col
0,1,09/15/1981,4,2,11/15/1998,1998,1,07/15/1997,11/15/1998,610.0,274.0,72.0,6.00,6.00,20.0,9701.0,274.0,"SALESPERSONS, N.E.C.",SALES OCCUPATIONS,"TECHNICAL, SALES, AND ADMINISTRATIVE SUPPORT O...",RETAIL BAKERIES,ACCOMODATIONS AND FOOD SERVICES,0.0,12.0,1.0,1
1,1,09/15/1981,4,2,12/15/1999,1999,1,11/15/1998,03/15/1999,610.0,274.0,89.0,6.25,6.25,30.0,9701.0,274.0,"SALESPERSONS, N.E.C.",SALES OCCUPATIONS,"TECHNICAL, SALES, AND ADMINISTRATIVE SUPPORT O...",RETAIL BAKERIES,ACCOMODATIONS AND FOOD SERVICES,0.0,12.0,1.0,2
2,1,09/15/1981,4,2,12/15/1999,1999,1,03/15/1999,12/15/1999,NaN,NaN,39.0,13.75,NaN,8.0,199902.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,12.0,1.0,3
3,1,09/15/1981,4,2,12/15/2000,2000,1,12/15/1999,12/15/2000,641.0,435.0,91.0,NaN,NaN,20.0,199902.0,435.0,WAITER/WAITRESS,"SERVICE OCCUPATIONS, EXCEPT PROTECTIVE AND HOU...",SERVICE OCCUPATIONS,EATING AND DRINKING PLACES,ACCOMODATIONS AND FOOD SERVICES,0.0,13.0,1.0,4
4,1,09/15/1981,4,2,12/15/2001,2001,1,12/15/2000,12/15/2001,641.0,435.0,140.0,13.75,213.75,30.0,199902.0,435.0,WAITER/WAITRESS,"SERVICE OCCUPATIONS, EXCEPT PROTECTIVE AND HOU...",SERVICE OCCUPATIONS,EATING AND DRINKING PLACES,ACCOMODATIONS AND FOOD SERVICES,0.0,14.0,1.0,5



Column names:
['PUBID_1997', 'DOB', 'SAMPLE_RACE_1997', 'SAMPLE_SEX_1997', 'Interview_Date', 'Year', 'Employed', 'StartDate', 'StopDate', 'IND', 'OCC', 'TENURE', 'HRLY_WAGE', 'HRLY_COMP', 'HRS_WRK', 'UID', 'Code_1990', 'Occupation', 'Occupation_Group', 'Occupation_Group2', 'Industry', 'Industry_Group', 'marital_status', 'HGC', 'Region', 'index_col']

Data types:
PUBID_1997             int64
DOB                   object
SAMPLE_RACE_1997       int64
SAMPLE_SEX_1997        int64
Interview_Date        object
Year                   int64
Employed               int64
StartDate             object
StopDate              object
IND                  float64
OCC                  float64
TENURE               float64
HRLY_WAGE            float64
HRLY_COMP            float64
HRS_WRK              float64
UID                  float64
Code_1990            float64
Occupation            object
Occupation_Group      object
Occupation_Group2     object
Industry              object
Industry_Group        obj

,0
HRLY_COMP,8.83
HRLY_WAGE,5.51
Occupation_Group,4.17
Occupation,4.17
Occupation_Group2,4.17
HRS_WRK,3.60
TENURE,3.19
Industry,2.06
Industry_Group,2.06
Code_1990,2.02



Working CSV saved to: /content/graduate-full.csv
Using file: /content/datathon_data/extracted/graduate-full.csv
Raw shape: (194545, 26)


,PUBID_1997,DOB,SAMPLE_RACE_1997,SAMPLE_SEX_1997,Interview_Date,Year,Employed,StartDate,StopDate,IND,OCC,TENURE,HRLY_WAGE,HRLY_COMP,HRS_WRK,UID,Code_1990,Occupation,Occupation_Group,Occupation_Group2,Industry,Industry_Group,marital_status,HGC,Region,index_col
0,1,09/15/1981,4,2,11/15/1998,1998,1,07/15/1997,11/15/1998,610.0,274.0,72.0,6.00,6.00,20.0,9701.0,274.0,"SALESPERSONS, N.E.C.",SALES OCCUPATIONS,"TECHNICAL, SALES, AND ADMINISTRATIVE SUPPORT O...",RETAIL BAKERIES,ACCOMODATIONS AND FOOD SERVICES,0.0,12.0,1.0,1
1,1,09/15/1981,4,2,12/15/1999,1999,1,11/15/1998,03/15/1999,610.0,274.0,89.0,6.25,6.25,30.0,9701.0,274.0,"SALESPERSONS, N.E.C.",SALES OCCUPATIONS,"TECHNICAL, SALES, AND ADMINISTRATIVE SUPPORT O...",RETAIL BAKERIES,ACCOMODATIONS AND FOOD SERVICES,0.0,12.0,1.0,2
2,1,09/15/1981,4,2,12/15/1999,1999,1,03/15/1999,12/15/1999,NaN,NaN,39.0,13.75,NaN,8.0,199902.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,12.0,1.0,3
3,1,09/15/1981,4,2,12/15/2000,2000,1,12/15/1999,12/15/2000,641.0,435.0,91.0,NaN,NaN,20.0,199902.0,435.0,WAITER/WAITRESS,"SERVICE OCCUPATIONS, EXCEPT PROTECTIVE AND HOU...",SERVICE OCCUPATIONS,EATING AND DRINKING PLACES,ACCOMODATIONS AND FOOD SERVICES,0.0,13.0,1.0,4
4,1,09/15/1981,4,2,12/15/2001,2001,1,12/15/2000,12/15/2001,641.0,435.0,140.0,13.75,213.75,30.0,199902.0,435.0,WAITER/WAITRESS,"SERVICE OCCUPATIONS, EXCEPT PROTECTIVE AND HOU...",SERVICE OCCUPATIONS,EATING AND DRINKING PLACES,ACCOMODATIONS AND FOOD SERVICES,0.0,14.0,1.0,5


Q1. Is there evidence of pay discrimination between male and female employees?

Yes. The notebook provides consistent evidence of a gender pay gap, and the result remains after adding increasingly rich controls.

Your main earnings model uses log hourly wage as the dependent variable and progressively adds controls for age, age squared, education, tenure, weekly hours, race, region, marital status, occupation, industry, and year fixed effects. This matches the assignment’s request to justify model choice and controls.

Based on the notebook results:

Model 1: female coefficient = -0.1140, about -10.77%

Model 2: female coefficient = -0.1182, about -11.15%

Model 3: female coefficient = -0.1007, about -9.58%

Interpretation: even after controlling for observable characteristics and then adding occupation and industry, women still earn about 9.6% less than men on average in the full model.

This is the strongest answer to Question 1:
there is clear evidence of a persistent gender wage gap in the sample, and the gap is not explained away by basic demographics, work intensity, education, occupation, industry, or time effects.

A good presentation sentence is:

The regression results show a statistically significant and economically meaningful negative female coefficient across all specifications, with the fully controlled model indicating that women earn about 9.6% less than comparable men.

Q2. Is there evidence that asking about prior salary negatively impacts female employees? Should a similar ban be implemented nationally?

The notebook supports an argument that reliance on prior salary can disadvantage women, although the evidence is indirect rather than strict causal proof.

This distinction is important because the assignment background explains that California banned salary-history questions because prior pay can perpetuate past discrimination.

Evidence from the notebook

The prior-pay models show that current wage is strongly tied to prior wage:

Model 4: ln_prior_wage = 0.5297, female = -0.0588

Model 5: ln_prior_wage = 0.4358, female = -0.0527

ln_prior_wage:female = -0.0011, p = 0.784

Interpretation:

Prior wage strongly predicts current wage. That is the most important finding.

Women still earn less even after prior wage is included.

The interaction term between prior wage and female is not statistically significant, so the slope linking prior wage to current wage is not meaningfully different for women versus men in this specification.

So the best answer is not “the notebook proves employers asking prior pay hurts women more in slope terms.”
The better answer is:

Women already have lower wages

Current wages are strongly anchored to prior wages

Therefore, compensation systems that rely on salary history can carry forward existing gender wage inequality

That is exactly the policy concern described in the assignment prompt.

Recommendation on a national ban

Yes, I would recommend a similar ban nationally, with a careful justification:

The evidence suggests that prior wage is a strong predictor of current wage, while women continue to face a negative wage differential. Because salary-history reliance can transmit earlier inequities into future pay, a national restriction on asking about prior salary is policy-reasonable.

At the same time, be methodologically honest:

The dataset does not directly observe whether employers asked about salary history, so the evidence supports the policy rationale but does not by itself establish a strict causal effect of employer questioning behavior.

That wording is strong and safe.

Q3. After the California law, would the relationship between prior pay and current pay weaken? Would it become zero?

Based on your notebook, the relationship would not be expected to fall to zero. In fact, your model results suggest it remains positive and strong.

From the notebook:

Model 6: ln_prior_wage = 0.4218

ln_prior_wage:post_2018 = 0.1169, statistically significant

Model 7: female:post_2018 = -0.0161, statistically significant

Model 8: ln_prior_wage:female:post_2018 = -0.0164, p = 0.288, not significant

How to interpret this carefully

The assignment asks whether you would expect the prior-current wage relationship to weaken after the law and whether it should be zero.

A strong answer is:

Conceptually, I would expect the relationship between prior pay and current pay to weaken if employers are no longer allowed to directly request salary history. However, I would not expect the relationship to become zero, because prior pay also reflects experience, occupation trajectory, industry sorting, bargaining history, and labor market position.

Your actual notebook results strengthen the second part:

the relationship remains strongly positive

the post-2018 interaction is not negative

so your analysis does not show that the relationship disappeared

Because your data are not a clean California-only treatment design, you should not over-interpret post-2018 as a pure legal effect. The safest presentation is:

The notebook does not provide evidence that the prior-pay relationship becomes zero after 2018. This is reasonable, since prior pay is also an indirect proxy for accumulated human capital and labor market sorting, not just employer salary-history usage.

Q4. Are there hidden or latent factors driving the observed differences?

Yes. Several latent factors could influence both gender and wages and therefore help explain the observed differences.

The strongest list to present is:

unobserved worker productivity

career interruptions

caregiving responsibilities and motherhood penalty

negotiation behavior

employer pay-setting policies

firm size

union coverage

local labor market conditions

job-switch timing

unobserved job quality

selection into employment

measurement error in hours worked or wages

A polished answer is:

Some of the observed gender differences may reflect unmeasured factors that are not fully captured in the dataset, including career interruptions, caregiving burden, firm compensation policies, negotiation dynamics, and local labor market conditions. These omitted factors may influence both prior wage and current wage, which means the estimated relationships should be interpreted as strong associations rather than complete causal effects.